# BÀI LAB THỰC HÀNH SQL VỚI SQLITE3 TRONG JUPYTER NOTEBOOK

## Mục tiêu bài lab
Sau khi hoàn thành bài lab này, bạn sẽ nắm vững cách sử dụng các câu lệnh SQL thuộc các nhóm:

- DDL (Data Definition Language): CREATE, ALTER, DROP
- DML (Data Manipulation Language): INSERT, UPDATE, DELETE
- DQL (Data Query Language): SELECT kết hợp với các mệnh đề WHERE, ORDER BY, LIMIT, v.v.
- Joins & Subqueries: INNER JOIN, LEFT JOIN, truy vấn con.
- Advanced: GROUP BY, HAVING, Hàm tổng hợp, UNION, VIEW, INDEX, Transaction.

## Phần 0: Khởi tạo môi trường

Chạy cell dưới đây để kết nối với database SQLite và tạo một hàm hỗ trợ hiển thị kết quả dưới dạng bảng (giống pandas) cho đẹp mắt.

In [2]:
import sqlite3
import pandas as pd

In [3]:
# Kết nối đến database (file 'lab_sql.db' sẽ được tạo ở thư mục hiện tại)
conn = sqlite3.connect('lab_sql.db')
cursor = conn.cursor()

In [4]:
# Hàm hỗ trợ chạy SQL và hiển thị kết quả dạng bảng đẹp mắt
def run_sql(sql):
    try:
        df = pd.read_sql_query(sql, conn)
        display(df)
    except Exception as e:
        print(f"Lỗi: {e}")

print("Kết nối thành công đến database!")

Kết nối thành công đến database!


## PHẦN 1: DDL (Data Definition Language)
Ngôn ngữ định nghĩa dữ liệu: Dùng để tạo, sửa đổi và xóa cấu trúc bảng.

### 1.1. Lệnh CREATE TABLE
Chúng ta sẽ tạo 2 bảng: Departments (Phòng ban) và Employees (Nhân viên).

In [8]:
cursor.execute('''
CREATE TABLE IF NOT EXISTS Departments (
    DeptID INTEGER PRIMARY KEY AUTOINCREMENT,
    DeptName TEXT NOT NULL,
    Location TEXT
);
''')

In [9]:
cursor.execute('''
CREATE TABLE IF NOT EXISTS Employees (
    EmpID INTEGER PRIMARY KEY AUTOINCREMENT,
    Name TEXT NOT NULL,
    Age INTEGER,
    Salary REAL,
    DeptID INTEGER,
    ManagerID INTEGER,
    HireDate TEXT,
    FOREIGN KEY (DeptID) REFERENCES Departments(DeptID)
);
''')
conn.commit()
print("Đã tạo bảng Departments và Employees thành công!")

Đã tạo bảng Departments và Employees thành công!


#### TODO 1: Tạo bảng mới
- Yêu cầu: Tạo một bảng tên là Projects có các cột: ProjectID (Khóa chính, tự tăng), ProjectName (Text, Not Null), Budget (Real), DeptID (Khóa ngoại tham chiếu đến bảng Departments).

In [10]:
cursor.execute('''
CREATE TABLE IF NOT EXISTS Projects (
    ProjectID INTEGER PRIMARY KEY AUTOINCREMENT,
    ProjectName TEXT NOT NULL,
    Budget REAL,
    DeptID INTEGER,
    FOREIGN KEY (DeptID) REFERENCES Departments(DeptID)
);
''')
conn.commit()
print("Đã tạo bảng Projects thành công!")

Đã tạo bảng Projects thành công!


### 1.2. Lệnh ALTER TABLE
Dùng để thêm hoặc sửa đổi cột.

In [11]:
# Thêm cột mới vào bảng Employees
cursor.execute('ALTER TABLE Employees ADD COLUMN Email TEXT;')
conn.commit()
run_sql("PRAGMA table_info(Employees);") # Xem cấu trúc bảng

,cid,name,type,notnull,dflt_value,pk
0,0,EmpID,INTEGER,0,None,1
1,1,Name,TEXT,1,None,0
2,2,Age,INTEGER,0,None,0
3,3,Salary,REAL,0,None,0
4,4,DeptID,INTEGER,0,None,0
5,5,ManagerID,INTEGER,0,None,0
6,6,HireDate,TEXT,0,None,0
7,7,Email,TEXT,0,None,0


#### TODO 2: Sửa đổi bảng
Yêu cầu: Thêm một cột Status (kiểu TEXT, mặc định là 'Active') vào bảng Projects vừa tạo.

In [13]:
cursor.execute('''
ALTER TABLE Projects 
ADD COLUMN status TEXT DEFAULT 'active';
''')
conn.commit()
print("Đã thêm cột status thành công!")

Đã thêm cột status thành công!


In [14]:
run_sql("PRAGMA table_info(Projects);") # Xem cấu trúc bảng

,cid,name,type,notnull,dflt_value,pk
0,0,ProjectID,INTEGER,0,None,1
1,1,ProjectName,TEXT,1,None,0
2,2,Budget,REAL,0,None,0
3,3,DeptID,INTEGER,0,None,0
4,4,status,TEXT,0,'active',0


## PHẦN 2: DML (Data Manipulation Language)
Ngôn ngữ thao tác dữ liệu: Thêm, Sửa, Xóa dòng dữ liệu.

### 2.1. Lệnh INSERT
Chèn dữ liệu vào bảng.

In [16]:
# Insert Departments
cursor.executemany('''
INSERT INTO Departments (DeptName, Location) VALUES (?, ?)
''', [
    ('Engineering', 'Floor 1'),
    ('Marketing', 'Floor 2'),
    ('Sales', 'Floor 3'),
    ('HR', 'Floor 4')
])
conn.commit()

In [18]:
# Insert Employees
cursor.executemany('''
INSERT INTO Employees (Name, Age, Salary, DeptID, ManagerID, HireDate) 
VALUES (?, ?, ?, ?, ?, ?)
''', [
    ('Nguyen Van A', 28, 15000000, 1, 0, '2020-05-15'),
    ('Le Thi B', 32, 20000000, 1, 1, '2018-01-10'),
    ('Tran Van C', 25, 12000000, 2, 0, '2022-08-01'),
    ('Pham Thi D', 40, 25000000, 3, 0, '2015-11-20'),
    ('Hoang Van E', 30, 18000000, 2, 3, '2019-06-05'),
    ('Vo Thi F', 26, 11000000, 3, 4, '2023-01-15'),
    ('Bui Van G', 35, 30000000, 1, 2, '2012-03-25')
])
conn.commit()

print("Đã chèn dữ liệu mẫu!")

Đã chèn dữ liệu mẫu!


In [22]:
run_sql("SELECT * FROM Departments;")

,DeptID,DeptName,Location
0,1,Engineering,Floor 1
1,2,Marketing,Floor 2
2,3,Sales,Floor 3
3,4,HR,Floor 4
4,5,Engineering,Floor 1
5,6,Marketing,Floor 2
6,7,Sales,Floor 3
7,8,HR,Floor 4


#### TODO 3: Thêm dữ liệu
Yêu cầu: Thêm 2 dòng dữ liệu vào bảng Projects (Project1: 'Website Redesign', Budget: 50000000, DeptID: 1; Project2: 'Q4 Campaign', Budget: 20000000, DeptID: 2).

In [23]:
#Insert Pojects
cursor.executemany('''
INSERT INTO Projects (ProjectName, Budget, DeptID)
VALUES (?,?,?)
''', [
    ('Website Redesign', 50000000, 1),
    ('Q4 Campaign', 20000000, 2)
])
conn.commit()

print("Đã chèn dữ liệu mẫu!")

Đã chèn dữ liệu mẫu!


In [24]:
run_sql("SELECT * FROM Projects;")

,ProjectID,ProjectName,Budget,DeptID,status
0,1,Website Redesign,50000000.0,1,active
1,2,Q4 Campaign,20000000.0,2,active


### 2.2. Lệnh UPDATE
Cập nhật dữ liệu đã có.

In [25]:
# Tăng lương 10% cho nhân viên thuộc phòng Engineering (DeptID = 1)
cursor.execute('''
UPDATE Employees 
SET Salary = Salary * 1.1 
WHERE DeptID = 1;
''')
conn.commit()
run_sql("SELECT Name, Salary FROM Employees WHERE DeptID = 1;")

,Name,Salary
0,Nguyen Van A,16500000.0
1,Le Thi B,22000000.0
2,Bui Van G,33000000.0


#### TODO 4: Cập nhật dữ liệu
Yêu cầu: Cập nhật Email cho nhân viên có tên 'Nguyen Van A' thành 'ana@company.com'.

In [26]:
cursor.execute('''
UPDATE Employees
SET Email = 'ana@company.com'
WHERE Name = 'Nguyen Van A';
''')
conn.commit()
run_sql("SELECT Name, Email FROM Employees WHERE Name = 'Nguyen Van A';")

,Name,Email
0,Nguyen Van A,ana@company.com


### 2.3. Lệnh DELETE
Xóa dòng dữ liệu.

In [28]:
# Xóa nhân viên có ID = 6 (Vo Thi F)
cursor.execute("DELETE FROM Employees WHERE EmpID = 6;")
conn.commit()
run_sql("SELECT * FROM Employees;")

,EmpID,Name,Age,Salary,DeptID,ManagerID,HireDate,Email
0,1,Nguyen Van A,28,16500000.0,1,0,2020-05-15,ana@company.com
1,2,Le Thi B,32,22000000.0,1,1,2018-01-10,None
2,3,Tran Van C,25,12000000.0,2,0,2022-08-01,None
3,4,Pham Thi D,40,25000000.0,3,0,2015-11-20,None
4,5,Hoang Van E,30,18000000.0,2,3,2019-06-05,None
5,7,Bui Van G,35,33000000.0,1,2,2012-03-25,None


#### TODO 5: Xóa dữ liệu
Yêu cầu: Xóa tất cả các dự án (Projects) có Budget nhỏ hơn 30000000.

In [29]:
cursor.execute("""
DELETE FROM Projects
WHERE Budget < 30000000;
""")

conn.commit()

run_sql("SELECT * FROM Projects;")

,ProjectID,ProjectName,Budget,DeptID,status
0,1,Website Redesign,50000000.0,1,active


## PHẦN 3: DQL (Data Query Language) - Cơ bản
Truy vấn và lọc dữ liệu.

### 3.1. SELECT, WHERE, ORDER BY, LIMIT

In [32]:
# Chọn tất cả nhân viên, sắp xếp theo lương giảm dần, giới hạn 3 người
run_sql('''
SELECT Name, Salary 
FROM Employees;
''')

,Name,Salary
0,Nguyen Van A,16500000.0
1,Le Thi B,22000000.0
2,Tran Van C,12000000.0
3,Pham Thi D,25000000.0
4,Hoang Van E,18000000.0
5,Bui Van G,33000000.0


In [33]:
# Chọn tất cả nhân viên, sắp xếp theo lương giảm dần, giới hạn 3 người
run_sql('''
SELECT *
FROM Employees;
''')

,EmpID,Name,Age,Salary,DeptID,ManagerID,HireDate,Email
0,1,Nguyen Van A,28,16500000.0,1,0,2020-05-15,ana@company.com
1,2,Le Thi B,32,22000000.0,1,1,2018-01-10,None
2,3,Tran Van C,25,12000000.0,2,0,2022-08-01,None
3,4,Pham Thi D,40,25000000.0,3,0,2015-11-20,None
4,5,Hoang Van E,30,18000000.0,2,3,2019-06-05,None
5,7,Bui Van G,35,33000000.0,1,2,2012-03-25,None


In [34]:
# Chọn tất cả nhân viên, sắp xếp theo lương giảm dần, giới hạn 3 người
run_sql('''
SELECT *
FROM Employees
ORDER BY Salary DESC 
''')

,EmpID,Name,Age,Salary,DeptID,ManagerID,HireDate,Email
0,7,Bui Van G,35,33000000.0,1,2,2012-03-25,None
1,4,Pham Thi D,40,25000000.0,3,0,2015-11-20,None
2,2,Le Thi B,32,22000000.0,1,1,2018-01-10,None
3,5,Hoang Van E,30,18000000.0,2,3,2019-06-05,None
4,1,Nguyen Van A,28,16500000.0,1,0,2020-05-15,ana@company.com
5,3,Tran Van C,25,12000000.0,2,0,2022-08-01,None


In [35]:
# Chọn tất cả nhân viên, sắp xếp theo lương giảm dần, giới hạn 3 người
run_sql('''
SELECT *
FROM Employees
ORDER BY Salary DESC 
LIMIT 3;
''')

,EmpID,Name,Age,Salary,DeptID,ManagerID,HireDate,Email
0,7,Bui Van G,35,33000000.0,1,2,2012-03-25,None
1,4,Pham Thi D,40,25000000.0,3,0,2015-11-20,None
2,2,Le Thi B,32,22000000.0,1,1,2018-01-10,None


In [36]:
# Chọn tất cả nhân viên, sắp xếp theo lương giảm dần, giới hạn 3 người
run_sql('''
SELECT *
FROM Employees
WHERE DeptID =1
ORDER BY Salary DESC 
LIMIT 3

''')

,EmpID,Name,Age,Salary,DeptID,ManagerID,HireDate,Email
0,7,Bui Van G,35,33000000.0,1,2,2012-03-25,None
1,2,Le Thi B,32,22000000.0,1,1,2018-01-10,None
2,1,Nguyen Van A,28,16500000.0,1,0,2020-05-15,ana@company.com


#### TODO 6: Truy vấn có điều kiện
Yêu cầu: Tìm tất cả nhân viên có tuổi lớn hơn 28, có DeptID là 1 hoặc 2 (Dùng IN hoặc OR). Hiển thị tên, tuổi và tên phòng ban.

In [37]:
run_sql('''
SELECT Name, Age, Deptid
FROM Employees
WHERE Age > 28 And DeptID IN (1, 2);
''')


,Name,Age,DeptID
0,Le Thi B,32,1
1,Hoang Van E,30,2
2,Bui Van G,35,1


In [38]:
run_sql("""
SELECT Name, Age, Deptid FROM Employees
WHERE Age > 28 AND (Deptid =1 OR Deptid =2)
""")

,Name,Age,DeptID
0,Le Thi B,32,1
1,Hoang Van E,30,2
2,Bui Van G,35,1


## PHẦN 4: HÀM TỔNG HỢP VÀ GROUP BY
### 4.1. COUNT, SUM, AVG, MAX, MIN với GROUP BY

In [39]:
# Thống kê số lượng nhân viên, lương trung bình, lương cao nhất theo từng phòng ban
run_sql('''
SELECT d.DeptName, 
       COUNT(e.EmpID) AS SoLuongNV,
       AVG(e.Salary) AS LuongTrungBinh,
       MAX(e.Salary) AS LuongCaoNhat
FROM Employees e
JOIN Departments d ON e.DeptID = d.DeptID
GROUP BY d.DeptName;
''')

,DeptName,SoLuongNV,LuongTrungBinh,LuongCaoNhat
0,Engineering,3,2.383333e+07,33000000.0
1,Marketing,2,1.500000e+07,18000000.0
2,Sales,1,2.500000e+07,25000000.0


In [40]:
# Thống kê số lượng nhân viên, lương trung bình, lương cao nhất theo toàn bộ công ty
run_sql('''
SELECT 
       COUNT(e.EmpID) AS SoLuongNV,
       AVG(e.Salary) AS LuongTrungBinh,
       MAX(e.Salary) AS LuongCaoNhat
FROM Employees e
JOIN Departments d ON e.DeptID = d.DeptID
''')

,SoLuongNV,LuongTrungBinh,LuongCaoNhat
0,6,2.108333e+07,33000000.0


## 4.2. Mệnh đề HAVING
HAVING giống như WHERE nhưng dùng cho các nhóm dữ liệu (sau GROUP BY).

In [41]:
# Chỉ lấy những phòng ban có Lương Trung Bình > 15000000
run_sql('''
SELECT d.DeptName, AVG(e.Salary) AS AvgSalary
FROM Employees e
JOIN Departments d ON e.DeptID = d.DeptID
GROUP BY d.DeptName
HAVING AvgSalary > 15000000;
''') 

,DeptName,AvgSalary
0,Engineering,2.383333e+07
1,Sales,2.500000e+07


#### TODO 8: Thống kê phức tạp
Yêu cầu: Đếm số lượng nhân viên cho từng người quản lý (ManagerID). Chỉ hiển thị những người quản lý đang quản lý từ 2 nhân viên trở lên.

In [46]:
# Chỉ lấy những phòng ban có Lương Trung Bình > 15000000
run_sql('''
SELECT e.ManagerID, COUNT(e.EmpID) AS Quantity
FROM Employees e
GROUP BY e.ManagerID
HAVING Quantity >= 2
''') 

,ManagerID,Quantity
0,0,3


##  PHẦN 5: JOINS & SUBQUERIES

### 5.1. INNER JOIN vs LEFT JOIN

In [49]:
# INNER JOIN: Chỉ lấy những phòng ban CÓ nhân viên
run_sql('''
SELECT d.DeptName, e.Name 
FROM Departments d
INNER JOIN Employees e ON d.DeptID = e.DeptID;
''')

,DeptName,Name
0,Engineering,Nguyen Van A
1,Engineering,Le Thi B
2,Marketing,Tran Van C
3,Sales,Pham Thi D
4,Marketing,Hoang Van E
5,Engineering,Bui Van G


In [50]:
# LEFT JOIN: Lấy TẤT CẢ phòng ban, phòng nào không có nhân viên thì Name sẽ là NULL
run_sql('''
SELECT d.DeptName, e.Name 
FROM Departments d
LEFT JOIN Employees e ON d.DeptID = e.DeptID;
''')

,DeptName,Name
0,Engineering,Bui Van G
1,Engineering,Le Thi B
2,Engineering,Nguyen Van A
3,Marketing,Hoang Van E
4,Marketing,Tran Van C
5,Sales,Pham Thi D
6,HR,None
7,Engineering,None
8,Marketing,None
9,Sales,None


### 5.2. Subquery (Truy vấn con)

In [51]:
# Tìm những nhân viên có lương cao hơn mức lương trung bình của toàn công ty
run_sql('''
SELECT AVG(Salary) FROM Employees
''')

,AVG(Salary)
0,2.108333e+07


In [52]:
# Tìm những nhân viên có lương cao hơn mức lương trung bình của toàn công ty
run_sql('''
SELECT Name, Salary 
FROM Employees 
WHERE Salary > (SELECT AVG(Salary) FROM Employees);
''')

,Name,Salary
0,Le Thi B,22000000.0
1,Pham Thi D,25000000.0
2,Bui Van G,33000000.0


#### TODO 10: Truy vấn con nâng cao
Yêu cầu: Tìm những nhân viên thuộc phòng ban có địa điểm ('Location') ở 'Floor 1'. (Không được dùng JOIN chính, phải dùng Subquery trong mệnh đề WHERE).